# 🃏 Poker AI v4 — Kaggle Trainer (Dataset Edition)

Ez a verzió a GitHub helyett a **Kaggle Datasets** szolgáltatást használja a modellek (checkpointok) tárolására és verziózására. 
A kód továbbra is a GitHubról jön, de a nagyméretű `.pth` fájlok itt, a Kaggle szerverein maradnak, így sokkal gyorsabb és stabilabb a folyamat.

### ⚠️ ELSŐ INDÍTÁS ELŐTTI TEENDŐK:

1. **API Kulcs beszerzése:**
   * Menj a Kaggle profilodba: *Settings -> Create New Token* (Ez letölt egy `kaggle.json` fájlt).
2. **Secrets beállítása:**
   * Itt a notebookban a bal oldali sávban kattints a **Secrets** (kulcs ikon) gombra.
   * Hozz létre kettőt: `KAGGLE_USERNAME` (a jsonből a username) és `KAGGLE_KEY` (a jsonből a key).
   * **Pipáld ki** mindkettő mellett a jelölőnégyzetet, hogy a notebook láthassa őket!
3. **Dataset létrehozása:**
   * A Kaggle felületén hozz létre egy új datasetet `SorosDb` néven (pl. tölts fel bele egy üres text fájlt, hogy engedje létrehozni). Enélkül nem tud hova pusholni a kód.

---
## 1. Lépés — Hitelesítés és Könyvtárak

In [ ]:
import os
import sys
import time
import json
import shutil
import threading
import subprocess

print('=' * 52)
print('  KAGGLE API HITELESÍTÉS')
print('=' * 52)

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")
    print("✅ Kaggle API hitelesítés beállítva a Secrets alapján.")
except Exception as e:
    print("❌ Hiba a hitelesítésnél!")
    print("👉 Kérlek, állítsd be a KAGGLE_USERNAME és KAGGLE_KEY secreteket a bal oldali menüben!")
    print("Részletek:", e)

# --- Alapvető útvonalak és nevek ---
DATASET_NAME = "SorosDb"
DATASET_ID = f"{os.environ.get('KAGGLE_USERNAME')}/{DATASET_NAME}"

CODE_DIR = '/kaggle/working/poker_ai'
OUTPUT_BASE = '/kaggle/working'
MODELS_DIR = os.path.join(OUTPUT_BASE, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)


---
## 2. Lépés — Kódbázis (GitHub) és Modellek (Dataset) szinkronizálása
Ez a lépés letölti a kódot a GitHubodról, majd ráhúzza az elmentett `.pth` modelleket a Kaggle Datasetedből.

In [ ]:
# 1. KÓD LETÖLTÉSE GITHUBRÓL (CSAK PULL/CLONE, PUSH NINCS TÖBBÉ)
REPO_URL = 'https://github.com/Mikidikilang/PokerAI.git'
REPO_BRANCH = 'main'

print("🔄 Kódbázis szinkronizálása a GitHubról...")
if os.path.exists(os.path.join(CODE_DIR, '.git')):
    subprocess.run(f'cd {CODE_DIR} && git fetch origin {REPO_BRANCH} && git reset --hard origin/{REPO_BRANCH}', shell=True)
else:
    subprocess.run(f'git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {CODE_DIR}', shell=True)

# Dependenciák telepítése
req_path = os.path.join(CODE_DIR, 'requirements.txt')
if os.path.exists(req_path):
    subprocess.run(f'pip install -q -r {req_path}', shell=True)
subprocess.run('pip install -q kaggle', shell=True) # Kaggle CLI csomag biztosítása
print("✅ Kód szinkronizálva.\n")

# 2. MODELLEK LETÖLTÉSE KAGGLE DATASETBŐL
print(f"🔄 Ellenőrzöm a korábbi modelleket a datasetben: {DATASET_ID}...")

# Legeneráljuk a Kaggle-nek a metadata fájlt, ez kötelező a későbbi feltöltéshez
metadata = {
  "title": "PokerAI Models",
  "id": DATASET_ID,
  "licenses": [{"name": "CC0-1.0"}]
}
with open(os.path.join(MODELS_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f)

# Letöltjük a legfrissebb dataset verziót
cmd = f"kaggle datasets download -d {DATASET_ID} -p {MODELS_DIR} --unzip"
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)

if r.returncode == 0:
    print("✅ Korábbi modellek sikeresen betöltve a working directory-ba!")
else:
    print("⚠️ Dataset nem letölthető vagy még üres. (Ha most hoztad létre a datasetet, ez normális!)")
    if "403" in r.stderr or "404" in r.stderr:
        print(f"👉 Kérlek ellenőrizd, hogy a dataset tényleg létezik-e ezen a néven: https://kaggle.com/{DATASET_ID}")


---
## 3. Lépés — Tréning Konfiguráció

In [ ]:
MODEL_NAME    = '2max'
NUM_PLAYERS   = 2
TRAINING_MODE = 'self_play'
EPISODES      = 50_000_000
SAVE_FREQ     = 500_000
TEST_HANDS    = 500
NUM_ENVS      = 256  # Állítsd be a GPU-d szerint

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from training.model_manager import ModelManager
mgr = ModelManager(OUTPUT_BASE)
mgr.ensure_model_dir(MODEL_NAME, NUM_PLAYERS)

cfg_data = mgr.load_config(MODEL_NAME)
cfg_data['num_players'] = NUM_PLAYERS
cfg_data['config'].update({
    'num_envs':           NUM_ENVS,
    'milestone_interval': SAVE_FREQ,
    'milestone_hands':    TEST_HANDS,
    'training_style':     TRAINING_MODE,
})
mgr.save_config(MODEL_NAME, cfg_data)

PTH_PATH = mgr.pth_path(MODEL_NAME)
print(f"✅ config.json megírva (mod: {TRAINING_MODE}).")
print(f"📍 Tervezett mentési útvonal: {PTH_PATH}")


---
## 4. Lépés — Tréning és Automatikus Kaggle Mentés
Itt indul el a tényleges tréning a `train.py` alapján, miközben egy háttérszál bizonyos időközönként feltölti a lementett fájlokat a Kaggle Datasetbe.

In [ ]:
# === HÁTTÉRSZÁL A DATASET FELTÖLTÉSHEZ ===
PUSH_INTERVAL_SEC = 300  # 5 percenként ellenőrzi
_stop_pusher = [False]
_last_mtime = [0.0]

UPLOAD_TMP = '/kaggle/working/upload_tmp'

def _prepare_upload_dir():
    """Csak a lényeges fájlokat másolja egy temp mappába feltöltéshez."""
    import shutil
    if os.path.exists(UPLOAD_TMP):
        shutil.rmtree(UPLOAD_TMP)
    os.makedirs(UPLOAD_TMP, exist_ok=True)
    # Fő checkpoint
    if os.path.exists(PTH_PATH):
        shutil.copy2(PTH_PATH, UPLOAD_TMP)
    # Config és napló
    for fname in ['config.json', 'naplo.json']:
        src = os.path.join(MODELS_DIR, MODEL_NAME, fname)
        if os.path.exists(src):
            shutil.copy2(src, UPLOAD_TMP)
    # Kaggle metadata (kötelező)
    meta_src = os.path.join(MODELS_DIR, 'dataset-metadata.json')
    if os.path.exists(meta_src):
        shutil.copy2(meta_src, UPLOAD_TMP)

def kaggle_pusher_thread():
    while not _stop_pusher[0]:
        time.sleep(PUSH_INTERVAL_SEC)
        if _stop_pusher[0]:
            break
        
        try:
            mtime = os.path.getmtime(PTH_PATH) if os.path.exists(PTH_PATH) else 0.0
            if mtime > _last_mtime[0]:
                print(f"\n[Auto-Save] 🚀 Új checkpoint észlelve! Feltöltés a datasetbe ({DATASET_ID})...", flush=True)
                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
                _prepare_upload_dir()
                cmd = f'kaggle datasets version -p {UPLOAD_TMP} -m "Auto update: {timestamp}"'
                r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
                
                if r.returncode == 0:
                    _last_mtime[0] = mtime
                    print("[Auto-Save] ✅ Dataset verzió sikeresen frissítve a Kaggle szerverein!", flush=True)
                else:
                    print(f"[Auto-Save] ❌ Hiba a feltöltésnél: {r.stderr or r.stdout}", flush=True)
        except Exception as e:
            print(f"\n[Auto-Save] Szál hiba: {e}", flush=True)

pusher = threading.Thread(target=kaggle_pusher_thread, daemon=True)
pusher.start()
print(f"⏱️ Háttér mentési szál elindítva ({PUSH_INTERVAL_SEC} másodperces intervallummal).")

# === TRÉNING FOLYAMAT INDÍTÁSA ===
cmd = [
    sys.executable,
    os.path.join(CODE_DIR, 'train.py'),
    '--model_name',  MODEL_NAME,
    '--output_dir',  OUTPUT_BASE,
    '--iterations', str(EPISODES),
]

print('\n' + '=' * 60)
print('  🚀 POKER AI TRÉNING INDUL')
print('=' * 60)

env = {**os.environ, 'PYTHONPATH': CODE_DIR, 'PYTHONUNBUFFERED': '1'}
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\n\n🛑 Tréning megszakítva (Ctrl+C). A végső mentés következik...')
    process.terminate()
    process.wait()
finally:
    # Szál leállítása
    _stop_pusher[0] = True
    
    # === VÉGSŐ FELTÖLTÉS ===
    print("\n🚀 Végső állapot feltöltése a Kaggle Datasetbe...")
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    _prepare_upload_dir()
    cmd = f'kaggle datasets version -p {UPLOAD_TMP} -m "Final Update: {timestamp}"'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if r.returncode == 0:
        print("✅ Végső dataset verzió sikeresen feltöltve!")
    else:
        print(f"❌ Hiba a végső feltöltésnél: {r.stderr or r.stdout}")
    print("\n🔚 Minden folyamat lezárult. A modellt biztonságban megtalálod a Datasets menüdben!")